# Notebook 5: Combine All Projects into Merged Datasets

This notebook merges the labeled comment CSVs from every project into three combined files, each saved to `data/`:

| File | Contents |
|---|---|
| `combined_commit_comments.csv` | All commit comment CSVs merged |
| `combined_pr_inline_comments.csv` | All PR inline comment CSVs merged |
| `combined_all_comments.csv` | Both sources merged together |

Each file has `text` and `polarity` columns (`0` = neutral, `1` = positive, `2` = negative) plus a `source` column (`commit_comment` or `pr_inline_comment`) so rows can be filtered after loading.

This structure lets anyone training the sentiment model using Kaiaulu's `vignette/sentiment_analysis.Rmd` pick their setup by changing a single `fread()` URL (One project, one source type, or everything combined).

### Before you start

The per-project CSVs in `data/commit comments/` and `data/PR inline comments/` must exist. These are produced by Notebook 4. If they are missing, run Notebook 4 for each project first.

Running this notebook will write three files into `data/`:
- `combined_commit_comments.csv`
- `combined_pr_inline_comments.csv`
- `combined_all_comments.csv`

### Step 1: Import dependencies

In [ ]:
from pathlib import Path

import pandas as pd

### Step 2: Configure paths

In [ ]:
REPO_ROOT = Path("..").resolve()

COMMIT_DIR = REPO_ROOT / "data" / "commit comments"
PR_DIR     = REPO_ROOT / "data" / "PR inline comments"

OUT_COMMIT  = REPO_ROOT / "data" / "combined_commit_comments.csv"
OUT_PR      = REPO_ROOT / "data" / "combined_pr_inline_comments.csv"
OUT_ALL     = REPO_ROOT / "data" / "combined_all_comments.csv"

print(f"Commit comments directory   : {COMMIT_DIR}")
print(f"PR inline comments directory: {PR_DIR}")
print(f"Output (commit only)        : {OUT_COMMIT}")
print(f"Output (PR inline only)     : {OUT_PR}")
print(f"Output (all combined)       : {OUT_ALL}")

### Step 3: Load commit comment CSVs

Read every commit comment CSV and tag each row with `source = "commit_comment"` so we can tell where it came from after merging.

In [ ]:
commit_files = sorted(COMMIT_DIR.glob("*.csv"))
print(f"Found {len(commit_files)} commit comment CSV files")

commit_frames = []
for f in commit_files:
    df = pd.read_csv(f)
    df["source"] = "commit_comment"
    commit_frames.append(df)

commit_combined = pd.concat(commit_frames, ignore_index=True)
print(f"Total commit comment rows: {len(commit_combined)}")
commit_combined.head(3)

### Step 4: Load and tag PR inline comment CSVs

Same as Step 3, but for PR inline comments.

In [ ]:
pr_files = sorted(PR_DIR.glob("*.csv"))
print(f"Found {len(pr_files)} PR inline comment CSV files")

pr_frames = []
for f in pr_files:
    df = pd.read_csv(f)
    df["source"] = "pr_inline_comment"
    pr_frames.append(df)

pr_combined = pd.concat(pr_frames, ignore_index=True)
print(f"Total PR inline comment rows: {len(pr_combined)}")
pr_combined.head(3)

### Step 5: Validate each combined dataset

Check that every row has a `text` and `polarity` value before saving, since those are required for training.

In [ ]:
def validate(df, label):
    missing_text     = df["text"].isna().sum()
    missing_polarity = df["polarity"].isna().sum()
    print(f"\n--- {label} ---")
    print(f"Total rows: {len(df)}")
    print(f"Polarity distribution:")
    print(df["polarity"].value_counts().sort_index()
          .rename({0: "0 (neutral)", 1: "1 (positive)", 2: "2 (negative)"}).to_string())
    if missing_text > 0 or missing_polarity > 0:
        print(f"WARNING: {missing_text} rows missing 'text', {missing_polarity} rows missing 'polarity'")
    else:
        print("All rows have 'text' and 'polarity'. Ready to save.")

combined_all = pd.concat([commit_combined, pr_combined], ignore_index=True)

validate(commit_combined, "Commit comments only")
validate(pr_combined,     "PR inline comments only")
validate(combined_all,    "All combined")

### Step 6: Save the three combined CSVs

Each file is saved to `data/`. Use any one of these as the `fread()` input in `sentiment_analysis.Rmd` depending on which setup you want:

```r
# One source type
train_df <- fread("data/combined_commit_comments.csv")
train_df <- fread("data/combined_pr_inline_comments.csv")

# Everything
train_df <- fread("data/combined_all_comments.csv")
```

In [ ]:
commit_combined.to_csv(OUT_COMMIT, index=False)
pr_combined.to_csv(OUT_PR, index=False)
combined_all.to_csv(OUT_ALL, index=False)

print(f"Saved {len(commit_combined):,} rows → {OUT_COMMIT.name}")
print(f"Saved {len(pr_combined):,} rows → {OUT_PR.name}")
print(f"Saved {len(combined_all):,} rows → {OUT_ALL.name}")